# Docker Redis & 기본 자료구조 (시연)

> **이전까지는 데이터를 '표(PostgreSQL)'에 담았습니다. 오늘은 성격이 다른 데이터베이스 — 표가 아닌 key-value·인메모리 저장소 Redis를 처음 만납니다.** PostgreSQL을 띄웠던 바로 그 Docker로 Redis도 한 줄에 띄우고(단, **볼륨은 생략** — 인메모리), redis-py로 접속해 **String·Hash·List와 TTL**을 손에 익힙니다. 이 재료들은 나중에 **캐시·세션·큐**의 부품이 됩니다.

## 차시 학습 목표
- `docker run`으로 `db-redis`(포트 6379·**볼륨 생략**)를 띄우고 **redis-py**로 접속·PING 합니다.
- **String·TTL**(SET/GET·INCR·자동 만료), **Hash·List**(필드 묶음·순서 목록)를 직접 다룹니다.
- 각 자료구조의 **용도**(String→캐시/카운터·Hash→세션·List→큐)를 짚고, 이어지는 개인실습에서 스스로 골라 쓸 준비를 합니다.

## 다루는 내용
- Docker로 Redis 띄우기 & 접속
- String & TTL — 가장 단순한 key-value
- Hash & List — 세션과 큐의 재료
- 자료구조 용도 요약

## 실행 안내
- 🐳 **선행 조건**: `db-redis` 컨테이너(`redis:8`)를 씁니다. 아래 준비 셀이 자동 확인합니다(이미 떠 있으면 확인만 — 재실행 안전). PostgreSQL 실습 때 띄운 `db-pg`(PostgreSQL 5432)는 **건드리지 않고** 나란히 공존합니다.
- 위에서 아래로 **순서대로** 실행하면 에러 없이 끝까지 돕니다. 각 모듈은 시작에서 실습 키를 `r.delete(...)`로 초기화해 **몇 번을 재실행해도 같은 결과**가 나옵니다.
- 결과값은 제작 Mac의 Redis 8.8.1에서 실측한 **확정값**입니다(Redis 자료구조는 결정적 → LLM 미사용).
- 🔓 로컬 Redis는 **무비밀번호**라 `.env` 설정이 필요 없습니다(PostgreSQL과 다른 점).

## 연결 준비 — db-redis 기동 확인 & redis-py 연결

**최초 1회 기동 명령**(터미널): `docker run -d --name db-redis -p 6379:6379 redis:8` ← PostgreSQL을 띄웠던 그 Docker로, **볼륨 옵션만 빼고** 한 줄. 아래 셀은 이미 만들어 둔 컨테이너를 **켜고 상태만 확인**합니다(재생성하지 않음).

In [1]:
# ✅ 포인트: Python용 Redis 드라이버 redis-py를 설치합니다.
# 지금까지의 노트북(2·3번)은 redis-cli를 subprocess로 실행하는 방식이었지만, 여기서는 redis-py라는
# 파이썬 전용 라이브러리를 써서 좀 더 "파이썬스러운" 방식(r.set(...), r.get(...) 같은 메서드 호출)으로 다룹니다.
# %pip install -q redis 에서 %는 Jupyter의 매직 명령어 기호이고, -q는 quiet(설치 로그를 줄여서 출력)의 줄임말입니다.
%pip install -q redis


Note: you may need to restart the kernel to use updated packages.


In [2]:
# 💡 사전 준비(재실행 안전): db-redis를 켜고(이미 켜져 있으면 그대로), docker ps로 db-pg·db-redis 둘 다 확인합니다.
# ⚠️ Jupyter의 ! 매직에서는 docker --format 의 Go 템플릿({{...}})을 쓰지 않습니다(중괄호가 깨짐 — 상태는 docker ps로만).
# 아래는 Windows(cmd.exe) 기준입니다 — Linux/macOS는 바로 아래 주석 처리된 줄을 대신 쓰세요.
!docker start db-redis >NUL 2>&1 && echo db-redis 실행 중
# ---- Linux/macOS ----
# !docker start db-redis >/dev/null 2>&1 && echo "db-redis 실행 중"
!docker ps


db-redis ���� ��
CONTAINER ID   IMAGE                     COMMAND                   CREATED       STATUS       PORTS                                                                                      NAMES
d1ffbe6b0a3c   redis:8                   "docker-entrypoint.s…"   5 hours ago   Up 5 hours   0.0.0.0:6379->6379/tcp, [::]:6379->6379/tcp                                                db-redis
470f1c8e37cf   neo4j:2026.06-community   "tini -g -- /startup…"   4 days ago    Up 5 hours   0.0.0.0:7474->7474/tcp, [::]:7474->7474/tcp, 0.0.0.0:7687->7687/tcp, [::]:7687->7687/tcp   db-neo4j
b9045c5dac14   pgvector/pgvector:pg17    "docker-entrypoint.s…"   6 days ago    Up 5 hours   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp                                                db-pg


In [3]:
# ✅ 포인트: 파이썬에서 redis-py로 연결합니다 — r.ping()이 True면 성공.
# 💡 decode_responses=True 고정: 결과가 str로 옵니다(안 주면 b'...' 같은 bytes). 무비밀번호라 .env 설정도 불필요.
import redis

# redis.Redis(...)는 Redis 서버에 접속하는 "연결 객체"를 만듭니다 — 이후 이 r 변수를 통해 모든 명령을 보냅니다.
# host/port: Redis가 실행 중인 주소와 포트(6379는 Redis의 기본 포트). db=0: Redis는 번호가 매겨진 여러 개의
# 독립된 저장 공간(0~15번)을 가질 수 있는데, 0번을 쓰겠다는 뜻입니다(특별한 이유가 없으면 보통 0번을 씁니다).
r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
print("PING →", r.ping())          # 기대: True
print("redis-py 버전 →", redis.__version__)  # __version__은 설치된 라이브러리 자체의 버전을 보여주는 관례적인 속성
# 📌 볼륨 생략 = 컨테이너를 지우면(docker rm) 데이터가 사라집니다(인메모리) — 학습용 설정이며, 운영에서는 영속성·복제를 켜서 씁니다.


PING → True
redis-py 버전 → 8.1.0


### 정리
- **핵심**: `docker run -d --name db-redis -p 6379:6379 redis:8` 한 줄로 Redis가 뜨고(볼륨 생략), `redis-py`(파이썬)로 접속합니다. `r.ping()` → `True`.
- **db-pg와 공존**: Redis(6379)는 PostgreSQL(5432) **옆에 나란히** 뜹니다 — "컨테이너는 여러 개를 동시에 격리 실행"(Docker 개념의 확장).
- **흔한 실수**: `decode_responses=True`를 빼면 결과가 `b'...'`(bytes)로 나와 한글·문자열 비교에서 헷갈립니다 — 이 과정은 **항상 True로 고정**합니다.

## String & TTL — 가장 단순한 key-value

- **String**은 Redis의 가장 기본 값 타입입니다 — `SET`으로 넣고 `GET`으로 꺼냅니다. 숫자면 `INCR`로 셉니다(카운터).
- **TTL**(수명)을 주면 시간이 지나 **자동으로 사라집니다** — 세션·캐시에 딱 맞는 성질.

In [4]:
# ✅ 포인트: SET으로 넣고 GET으로 꺼냅니다 — key 하나로 value를 순식간에.
r.delete("greeting", "count", "temp")   # 💡 실습 키 초기화(멱등 — 재실행 안전)
# r.set(key, value): redis-cli의 `SET key value`와 같은 명령을 파이썬 메서드로 호출하는 방식입니다
# (redis-py는 Redis 명령어를 대부분 소문자 메서드 이름으로 그대로 제공합니다 — SET -> r.set, GET -> r.get).
r.set("greeting", "안녕하세요")
print("GET greeting →", r.get("greeting"))   # 기대: '안녕하세요' (decode된 str)


GET greeting → 안녕하세요


In [5]:
# ✅ 포인트: 숫자 값은 INCR로 1씩 증가 — 방문자 수·조회수 같은 카운터에 딱.
r.set("count", 0)
# incr(key)는 그 키의 값을 정수로 취급해 1 증가시킵니다 — "값을 GET해서 +1 한 뒤 다시 SET"하는 것을
# 한 번의 명령으로 안전하게 처리해줍니다(동시에 여러 요청이 몰려도 값이 꼬이지 않는 원자적 연산입니다).
r.incr("count")
r.incr("count")
print("count →", r.get("count"))   # 기대: '2'


count → 2


In [6]:
# ✅ 포인트: TTL(수명) — ex=초를 주면 그 시간 뒤 자동으로 사라집니다.
# 🆕 최신 문법: setex()는 deprecated → 항상 r.set(key, value, ex=초) 사용.
import time
# ex=2는 "EXpire 2초" — set() 호출 시점부터 2초 뒤 Redis가 이 키를 자동으로 지웁니다(4교시 run_redis 예시의
# "ex", "1800"과 같은 개념을, redis-py에서는 이렇게 키워드 인자로 넘깁니다).
r.set("temp", "곧 사라짐", ex=2)          # 2초 수명
# ttl(key)은 "이 키가 몇 초 뒤에 사라지는지" 남은 시간(초)을 반환합니다(TTL = Time To Live).
print("설정 직후 TTL →", r.ttl("temp"))   # 기대: 2 (또는 그 이하)
print("2초 전 값 →", r.get("temp"))       # 기대: '곧 사라짐'
time.sleep(2.2)                            # 💡 만료 관찰용 대기(2.2초 1회만 — 총 실행 시간 관리)
print("2.2초 후 값 →", r.get("temp"))      # 기대: None (자동 만료로 사라짐)


설정 직후 TTL → 2
2초 전 값 → 곧 사라짐
2.2초 후 값 → None


### 정리
- **핵심**: `SET`/`GET`=기본 저장·조회, `INCR`=카운터, **`set(k, v, ex=초)`=TTL(자동 만료)**. 시간이 지나면 키가 스스로 사라집니다.
- **🆕 최신 문법**: `setex()`는 **deprecated** → `r.set(key, value, ex=초)`로 통일합니다.
- **흔한 실수**: TTL을 안 주면 키는 **영원히** 남습니다 — 캐시·세션은 반드시 수명을 줘 메모리를 아낍니다.

## Hash & List — 세션과 큐의 재료

- **Hash**는 한 key 안에 **여러 필드-값**을 담습니다(이름·나이·도시…) — **세션**에 딱 맞는 모양.
- **List**는 **순서 있는 목록**입니다 — `LPUSH`로 넣고 `BRPOP`으로 꺼내면 **먼저 넣은 것이 먼저**(FIFO) → **큐**의 재료.

In [7]:
# ✅ 포인트: Hash = 한 key(user:demo)에 여러 필드 → 세션에 딱 맞는 모양.
r.delete("user:demo")   # 💡 멱등 초기화
# hset(key, mapping={...}): mapping 매개변수에 파이썬 딕셔너리를 통째로 넘기면, 그 딕셔너리의 모든
# 필드-값 쌍을 한 번에 저장합니다(2교시/4교시에서 본 run_redis("hset", key, "필드1", "값1", ...) 방식과
# 같은 결과지만, redis-py에서는 딕셔너리 하나로 더 간결하게 표현할 수 있습니다).
r.hset("user:demo", mapping={"name": "홍길동", "age": "30", "city": "서울", "job": "개발자"})
print("HGETALL →", r.hgetall("user:demo"))     # hgetall: 이 Hash의 모든 필드-값을 딕셔너리로 통째로 반환 # 기대: 4필드 dict
print("HLEN →", r.hlen("user:demo"))           # hlen: 이 Hash에 필드가 몇 개 있는지(길이) 반환 # 기대: 4
# hincrby(key, 필드, 증가량): Hash 안의 특정 필드 값을 정수로 취급해 증가시킵니다(위 incr의 Hash 버전).
print("HINCRBY age +1 →", r.hincrby("user:demo", "age", 1))   # 기대: 31 (정수 필드만 가능)


HGETALL → {'name': '홍길동', 'age': '30', 'city': '서울', 'job': '개발자'}
HLEN → 4
HINCRBY age +1 → 31


In [ ]:
# ✅ 포인트: List = 순서 목록. LPUSH(왼쪽 삽입) + BRPOP(오른쪽에서 꺼냄) = FIFO(먼저 넣은 것 먼저). \
#자료구조 중 큐와 원리가 동일함
r.delete("tasks")   # 💡 멱등 초기화
# lpush(key, *values): 왼쪽(맨 앞)에 값을 추가 — t1, t2, t3 순서로 호출하면 t3가 맨 왼쪽(가장 최근)에 위치합니다.
r.lpush("tasks", "t1", "t2", "t3")             # lpush는 새로운 값을 왼쪽에 채운다는 뜻
print("LRANGE 전체 →", r.lrange("tasks", 0, -1))       # 기대: ['t3','t2','t1'] (LPUSH라 역순)
print("LLEN →", r.llen("tasks"))                        # 기대: 3
# brpop(key, timeout=초): 오른쪽(맨 뒤)에서 값을 하나 꺼내며 제거합니다(pop). 앞에 B(Blocking)가 붙은 이유는
# 리스트가 비어있으면 값이 생길 때까지 최대 timeout초만큼 "기다리기" 때문입니다(일반 rpop은 비어있으면 즉시 None).
# 정리하면: 맨 왼쪽에 계속 쌓고(LPUSH) 맨 오른쪽에서 꺼내면(BRPOP) -> 가장 먼저 넣은 값(t1)이 가장 먼저 나옵니다
# -> FIFO(First In, First Out, 먼저 들어온 게 먼저 나감) 구조가 되며, 이는 "작업 대기열(큐)" 구현에 흔히 쓰입니다.
print("BRPOP(맨 뒤=먼저 넣은 것) →", r.brpop("tasks", timeout=1))   # 기대: ('tasks','t1') = FIFO


LRANGE 전체 → ['t3', 't2', 't1']
LLEN → 3
BRPOP(맨 뒤=먼저 넣은 것) → ('tasks', 't1')


### 정리
- **핵심**: **Hash=필드 묶음(→세션)**, **List=순서 목록(→큐)**. `LPUSH`(넣기)+`BRPOP`(꺼내기)=**FIFO**.
- **흔한 실수**: Hash 값은 전부 **문자열**로 저장됩니다(나이도 `"30"`). `HINCRBY`는 **정수처럼 보이는 필드**에만 됩니다.

### 자료구조 용도 요약

| 자료구조 | 대표 명령 | 성질 | 이후 쓰임 |
|----------|-----------|------|-----------------|
| **String** | `SET`/`GET`·`INCR` | 문자열·숫자 | 캐시·카운터 |
| **Hash** | `HSET`·`HGETALL`·`HINCRBY` | 한 key에 여러 필드 | **세션**(대화 상태) |
| **List** | `LPUSH`·`BRPOP` | 순서 목록(FIFO) | **큐**(작업 대기) |

## 마무리 — 오늘 익힌 재료와 다음 단계

Docker로 Redis를 띄우고(설치 0·볼륨 생략), 세 자료구조를 손에 익혔습니다.
- **연결**: `docker run` 한 줄 → `redis-py`로 접속(`r.ping()` → `True`), db-pg와 **공존**
- **String·TTL**: `SET`/`GET`·`INCR`·`set(ex=초)`로 **자동 만료**
- **Hash·List**: 필드 묶음(→세션)·순서 목록(→큐·`LPUSH`+`BRPOP`=FIFO)

**➡️ 다음(개인실습)**: 방금 익힌 세 자료구조로 **한빛은행 창구 운영 데이터**를 직접 다뤄봅니다 — 어떤 데이터에 어떤 자료구조가 맞는지 **스스로 고르는** 연습입니다.

## 종합 연습 — 쇼핑몰의 재고·타임세일·장바구니·최근 본 상품

오늘 배운 **String · TTL · Hash · List**를 쇼핑몰 상황에 그대로 붙여 봅니다. `db-redis` 컨테이너가 실행 중이어야 해요(위 연결 셀의 `r`을 그대로 씁니다).

정답은 맨 아래 **정답 모음**에 한꺼번에 있어요.

In [ ]:
# 🧪 Q1 — 상품 1번(노트북)의 재고 5를 String으로 저장하고 읽어 보세요 (key: stock:1)
# 💡 힌트: r.set → r.get — decode_responses=True 라 '5'(str)로 옵니다
# 👇 아래 빈 셀에 직접 작성해 보세요

In [10]:
r.set("stock:1", 5)
print("재고: ", r.get("stock:1"))

재고:  5


In [ ]:
# 🧪 Q2 — 상품 2번의 타임세일가 29900을 30초 수명으로 저장하고, 남은 수명과 값을 확인하세요 (key: sale:2)
# 💡 힌트: r.set(key, value, ex=30) → r.ttl → r.get (setex는 deprecated!)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [15]:
r.set("sale:2", 29900, ex=30)
print("남은 수명: ", r.ttl("sale:2"), "초")
print("세일가: ", r.get("sale:2"))

남은 수명:  30 초
세일가:  29900


In [11]:
# 🧪 Q3 — 장바구니를 Hash로: cart:1 에 {상품2: 1개, 상품6: 2개}를 담고, 상품2 수량을 1 올린 뒤 전체를 확인하세요
# 💡 힌트: r.delete로 초기화 → r.hset(mapping={...}) → r.hincrby → r.hgetall
# 👇 아래 빈 셀에 직접 작성해 보세요

In [19]:
r.delete("cart:1")
r.hset("cart:1", mapping={"2": "1", "6": "2"})
print("상품2 수량 +1 → ", r.hincrby("cart:1", "2", 1))
r.hgetall("cart:1")

상품2 수량 +1 →  2


{'2': '2', '6': '2'}

In [ ]:
# 🧪 Q4 — 최근 본 상품을 List로: viewed:1 에 상품 1 → 3 → 2 순서로 '본' 기록을 남기고, 최근 본 순서로 꺼내 보세요
# 💡 힌트: r.delete → 본 순서대로 r.lpush 세 번 → r.lrange(key, 0, -1) — 왼쪽 삽입이라 최근 것이 맨 앞!
# 👇 아래 빈 셀에 직접 작성해 보세요

In [20]:
r.delete("viewed:1")
r.lpush("viewed:1", "1", "3", "2")
r.lrange("viewed:1", 0, -1)

['2', '3', '1']

### 정답 모음 — 직접 푼 뒤에 확인하세요

In [ ]:
# ── Q1 정답 ──
# r.set("stock:1", 5)
# print("재고 →", r.get("stock:1"))
# # → 재고 → 5   (decode_responses=True 라 str '5' — 숫자 연산하려면 int() 변환)

In [ ]:
# ── Q2 정답 ──
# r.set("sale:2", 29900, ex=30)
# print("남은 수명 →", r.ttl("sale:2"), "초")
# print("세일가 →", r.get("sale:2"))
# # → 남은 수명 → 30 초 (또는 그 이하) / 세일가 → 29900
# # 💡 30초 뒤 다시 r.get("sale:2") 하면 None — 세일이 저절로 끝납니다

In [ ]:
# ── Q3 정답 ──
# r.delete("cart:1")
# r.hset("cart:1", mapping={"2": "1", "6": "2"})
# print("상품2 +1 →", r.hincrby("cart:1", "2", 1))
# print("장바구니 →", r.hgetall("cart:1"))
# # → 상품2 +1 → 2 / 장바구니 → {'2': '2', '6': '2'}

In [ ]:
# ── Q4 정답 ──
# r.delete("viewed:1")
# r.lpush("viewed:1", 1)
# r.lpush("viewed:1", 3)
# r.lpush("viewed:1", 2)
# print("최근 본 순서 →", r.lrange("viewed:1", 0, -1))
# print("개수 →", r.llen("viewed:1"))
# # → 최근 본 순서 → ['2', '3', '1'] / 개수 → 3
# # 💡 마지막에 본 상품 2가 맨 앞 — '최근 본 상품' UI가 딱 이 모양입니다